In [3]:
# Cell 1 - Import libraries for both models (Prophet for interpretable
# time-series forecasting, XGBoost for feature-rich gradient boosting),
# plus MAPE for comparing their accuracy on the same footing

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from prophet import Prophet
import xgboost as xgb
from sklearn.metrics import mean_absolute_percentage_error

d:\projects\Rossmann Store Sales\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Importing plotly failed. Interactive plots will not work.


In [4]:
# Cell 2 - Load the cleaned dataset 

df = pd.read_csv('../data/processed/rossmann_cleaned.csv', parse_dates=['Date'])

print('\nRows and Columns in dataset:')
print('-' * 50)
print(df.shape)

print('\nSummary:')
print('-' * 50)
df.info()


Rows and Columns in dataset:
--------------------------------------------------
(844338, 21)

Summary:
--------------------------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 844338 entries, 0 to 844337
Data columns (total 21 columns):
 #   Column                     Non-Null Count   Dtype         
---  ------                     --------------   -----         
 0   Store                      844338 non-null  int64         
 1   DayOfWeek                  844338 non-null  int64         
 2   Date                       844338 non-null  datetime64[us]
 3   Sales                      844338 non-null  int64         
 4   Customers                  844338 non-null  int64         
 5   Open                       844338 non-null  int64         
 6   Promo                      844338 non-null  int64         
 7   StateHoliday               844338 non-null  str           
 8   SchoolHoliday              844338 non-null  int64         
 9   StoreType                  844338 no

In [5]:
# Cell 3 - Aggregate to one row per day (chain-wide), matching what Prophet
# requires: exactly one target value per timestamp. Sales is summed (total
# chain revenue that day); Promo is averaged (fraction of open stores running a promo that day)

df_prophet = df.groupby('Date').agg({
    'Sales': 'sum',
    'Promo': 'mean'
}).reset_index()

df_prophet = df_prophet.rename(columns={'Date': 'ds', 'Sales': 'y'})
df_prophet.head()

,ds,y,Promo
0,2013-01-01,97235,0.0
1,2013-01-02,6949829,0.0
2,2013-01-03,6347820,0.0
3,2013-01-04,6638954,0.0
4,2013-01-05,5951593,0.0


In [6]:
# Cell 4 - Chronological train/validation split: last 12 weeks (84 days)
# held out as validation, matching the 12-week forecast horizon. Must NOT
# shuffle — a random split would leak future dates into training, which
# is meaningless for a real forecast that only ever knows the past

horizon_days = 84

train = df_prophet.iloc[:-horizon_days]
val = df_prophet.iloc[-horizon_days:]

print(train.shape, val.shape)

(858, 3) (84, 3)


In [7]:
# Cell 5 - Initialize Prophet with yearly and weekly seasonality (both
# verified real patterns in this data), daily seasonality off (data has
# no sub-day granularity to model), plus Promo as a regressor per the
# decision to strengthen Prophet with promo signal

prophet_model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False
)
prophet_model.add_regressor('Promo')

In [8]:
# Cell 6 - Fit Prophet on the training data 

prophet_model.fit(train)

09:01:04 - cmdstanpy - INFO - Chain [1] start processing
09:01:05 - cmdstanpy - INFO - Chain [1] done processing


In [9]:
# Cell 7 - Build the 84-day future dataframe (aligns exactly with val's
# date range) and attach the real Promo values from val, since the
# Promo regressor requires a value for every date being predicted

future = prophet_model.make_future_dataframe(periods=84)        # This also contains the training dates, so we need to add the Promo values for those too
future['Promo'] = pd.concat([train['Promo'], val['Promo']]).values

print('\nRows and Columns in future dataset:')
print('-' * 50)
print(future.shape)

print('\nLast 5 rows in future dataset:')
print('-' * 50)
future.tail()


Rows and Columns in future dataset:
--------------------------------------------------
(942, 2)

Last 5 rows in future dataset:
--------------------------------------------------


,ds,Promo
937,2015-07-27,1.0
938,2015-07-28,1.0
939,2015-07-29,1.0
940,2015-07-30,1.0
941,2015-07-31,1.0


In [10]:
# Cell 8 - Generate predictions for all 942 dates (858 training + 84 future),
# including yhat (point forecast) and yhat_lower/yhat_upper (confidence interval)

forecast = prophet_model.predict(future)

# yhat → Predicted
# yhat_lower → Lower bound
# yhat_upper → Upper bound

# yhat_lower / yhat_upper → the confidence interval around that guess. 
# Prophet is explicitly saying "I predict this value, but I'm not certain — 
# the real value could reasonably fall anywhere between yhat_lower and yhat_upper."
forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(10)

,ds,yhat,yhat_lower,yhat_upper
932,2015-07-22,5.576894e+06,3.798636e+06,7.418942e+06
933,2015-07-23,5.340971e+06,3.494504e+06,7.113759e+06
934,2015-07-24,5.761380e+06,3.910273e+06,7.616699e+06
935,2015-07-25,6.266129e+06,4.369969e+06,8.155108e+06
936,2015-07-26,1.887711e+05,-1.618581e+06,2.030982e+06
937,2015-07-27,9.581676e+06,7.729834e+06,1.147900e+07
938,2015-07-28,8.668637e+06,6.733543e+06,1.050657e+07
939,2015-07-29,8.173203e+06,6.285465e+06,1.002214e+07
940,2015-07-30,7.930373e+06,6.190945e+06,9.729158e+06
941,2015-07-31,8.343742e+06,6.470166e+06,1.016517e+07


In [11]:
# Cell 9 - Merge Prophet's validation-period predictions with the real
# known values, so predicted and actual sit in the same dataframe for a row-by-row comparison

forecast_val = forecast.tail(84)[['ds', 'yhat']]
val_actual = val[['ds', 'y']]

comparison = forecast_val.merge(val_actual, on='ds')
pd.set_option('display.float_format', '{:.2f}'.format)
comparison.head()

,ds,yhat,y
0,2015-05-09,6403791.87,7157061
1,2015-05-10,338739.06,251720
2,2015-05-11,7142639.03,6732629
3,2015-05-12,6238668.79,6686277
4,2015-05-13,5750441.81,8147927


In [12]:
# Cell 10 - Compute MAPE on the validation set: the real, honest test of
# forecast accuracy, since these 84 days were never seen during training

mape = mean_absolute_percentage_error(comparison['y'], comparison['yhat'])
print(f"Prophet MAPE: {mape:.4f} ({mape*100:.2f}%)")

Prophet MAPE: 0.6376 (63.76%)


## Investigating a High Baseline MAPE

The baseline Prophet forecast returned an unexpectedly high validation MAPE. 
Sorting individual daily errors showed two extreme outliers — 2015-05-14 and 
2015-05-25 — each over 1800% error. Tracing back to the raw data confirmed 
why: on both dates, over 97% of stores were closed chain-wide, consistent 
with German public holidays the model was never told about. Next: build a 
second Prophet model with Germany's holiday calendar added, and compare its 
MAPE directly against this baseline.

In [16]:
# Cell 11 - Rename as Baseline MAPE: Prophet's accuracy with no holiday
# awareness, kept and reported honestly rather than discarded, since the
# two extreme errors on 2015-05-14/05-25 have a confirmed root cause
# (near-total chain-wide shutdown, likely German public holidays)

baseline_mape = mean_absolute_percentage_error(comparison['y'], comparison['yhat'])
print(f"Baseline Prophet MAPE: {baseline_mape*100:.2f}%")

Baseline Prophet MAPE: 63.76%


## Why a New Model Object, Not a Modification to the Original

`holiday_model` is built as a separate object rather than adding holidays to 
`prophet_model` directly. Modifying `prophet_model` in place would require 
refitting it, which would overwrite the already-fitted baseline — losing the 
ability to compare "before" and "after" once both models are trained. Keeping 
them separate preserves the baseline exactly as reported, for a fair, 
reproducible comparison.

In [18]:
# Cell 12 - Build a second Prophet model, identical setup, plus Germany's
# public holidays, to test whether the two extreme errors were genuinely
# caused by unmodeled holiday shutdowns

holiday_model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False
)
holiday_model.add_country_holidays(country_name='DE')
holiday_model.add_regressor('Promo')
holiday_model.fit(train)

09:13:27 - cmdstanpy - INFO - Chain [1] start processing
09:13:27 - cmdstanpy - INFO - Chain [1] done processing


In [19]:
# Cell 13 - Generate holiday-aware forecast and compute its MAPE the same
# way as baseline, for a direct apples-to-apples comparison

future_holiday = holiday_model.make_future_dataframe(periods=84)
future_holiday['Promo'] = pd.concat([train['Promo'], val['Promo']]).values

forecast_holiday = holiday_model.predict(future_holiday)

comparison_holiday = forecast_holiday.tail(84)[['ds', 'yhat']].merge(val[['ds', 'y']], on='ds')
holiday_mape = mean_absolute_percentage_error(comparison_holiday['y'], comparison_holiday['yhat'])

print(f"Baseline Prophet MAPE:      {baseline_mape*100:.2f}%")
print(f"Holiday-aware Prophet MAPE: {holiday_mape*100:.2f}%")

Baseline Prophet MAPE:      63.76%
Holiday-aware Prophet MAPE: 32.89%


## Result: Holiday Awareness Nearly Halves MAPE

Adding Germany's public holiday calendar dropped validation MAPE from 63.76% 
to 32.89%. This confirms the two extreme-error dates were genuinely caused by 
unmodeled holiday shutdowns, not a general modeling failure — but 32.89% is 
still high in absolute terms. Next: check whether this remaining error is 
still concentrated in a small number of outlier days, or now spread more 
evenly across the validation period.

In [20]:
# Cell 14 - Re-run the per-row error breakdown on the holiday-aware model,
# to check whether the remaining MAPE is concentrated in a few outliers
# (like baseline was) or genuinely spread across most days

comparison_holiday['pct_error'] = abs(comparison_holiday['y'] - comparison_holiday['yhat']) / comparison_holiday['y'] * 100
comparison_holiday.sort_values('pct_error', ascending=False).head(10)

,ds,yhat,y,pct_error
1,2015-05-10,932227.27,251720,270.34
43,2015-06-21,863239.06,249849,245.50
36,2015-06-14,910798.51,269533,237.92
8,2015-05-17,855602.07,255680,234.64
29,2015-06-07,762952.31,262497,190.65
5,2015-05-14,796628.18,289247,175.41
15,2015-05-24,683906.74,261385,161.65
26,2015-06-04,8747924.25,3534231,147.52
50,2015-06-28,626252.83,262669,138.42
22,2015-05-31,630025.98,278690,126.07


## Testing a Higher Weekly-Seasonality Fourier Order

Even after adding holiday awareness, the largest remaining validation errors 
were concentrated on Sundays specifically — consistent with the sharp 
Sunday sales drop found in Notebook 1, which Prophet's default weekly 
seasonality (a smooth, low-order curve) may not capture sharply enough. 
Testing a higher Fourier order (10 instead of the default 3) lets the 
weekly component bend more steeply, to check whether that closes the gap. 
Holiday awareness and the Promo regressor are kept unchanged from the 
previous model, so any MAPE difference is attributable to this one change.

In [27]:
# Cell 15 - Tuned Prophet: adds multiplicative seasonality mode and
# DayOfWeek as a direct regressor, alongside the holiday calendar and
# Promo regressor already proven useful

# Add DayOfWeek as a regressor(feature) to both train and val, since it is a feature that will be used in the model
train['DayOfWeek'] = train['ds'].dt.dayofweek
val['DayOfWeek'] = val['ds'].dt.dayofweek

tuned_model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=10,
    daily_seasonality=False,
    seasonality_mode='multiplicative'
)
tuned_model.add_country_holidays(country_name='DE')
tuned_model.add_regressor('Promo')
tuned_model.add_regressor('DayOfWeek')
tuned_model.fit(train)

10:28:58 - cmdstanpy - INFO - Chain [1] start processing
10:28:58 - cmdstanpy - INFO - Chain [1] done processing


In [28]:
# Cell 16 - Generate the tuned model's forecast and MAPE

future_tuned = tuned_model.make_future_dataframe(periods=84)
future_tuned['Promo'] = pd.concat([train['Promo'], val['Promo']]).values
future_tuned['DayOfWeek'] = future_tuned['ds'].dt.dayofweek

forecast_tuned = tuned_model.predict(future_tuned)

comparison_tuned = forecast_tuned.tail(84)[['ds', 'yhat']].merge(val[['ds', 'y']], on='ds')
tuned_mape = mean_absolute_percentage_error(comparison_tuned['y'], comparison_tuned['yhat'])

print(f"Baseline Prophet MAPE:      {baseline_mape*100:.2f}%")
print(f"Holiday-aware Prophet MAPE: {holiday_mape*100:.2f}%")
print(f"Tuned Prophet MAPE:         {tuned_mape*100:.2f}%")

Baseline Prophet MAPE:      63.76%
Holiday-aware Prophet MAPE: 32.89%
Tuned Prophet MAPE:         22.48%


## Prophet Section — Final State

Final Prophet MAPE: 22.48%. Progression: 63.76% (baseline) → 32.89% 
(holiday-aware) → 22.48% (multiplicative seasonality + DayOfWeek regressor). 
The holiday fix resolved two extreme single-day errors caused by near-total 
chain-wide shutdowns; the final combination of multiplicative seasonality and an 
explicit DayOfWeek regressor produced the largest further improvement, 
suggesting Prophet's default additive weekly curve underrepresented how 
sharply Sunday sales drop relative to the rest of the week. This is used 
as Prophet's final benchmark to compare against XGBoost.